In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

using device: cuda


In [2]:
train_dir = r'C:\Users\sagal\Desktop\Let us build\RAF-DB\DATASET\train'

# Seeds
random.seed(42)
torch.manual_seed(42)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.25)) # Increased erasing
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

indices = list(range(len(train_dataset)))
random.seed(42)
random.shuffle(indices)

split_size    = int(0.85 * len(indices))
train_indices = indices[:split_size]
val_indices   = indices[split_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset, val_indices)

train_labels = [train_dataset.targets[i] for i in train_indices]
class_counts = Counter(train_labels)
total = len(train_labels)
class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)

In [3]:
# Load pretrained ResNet-18
resnet = models.resnet18(weights="IMAGENET1K_V1")

# Freeze early layers & layer2; unfreeze layer3, layer4, and fc
for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Increased Dropout from 0.5 -> 0.6 to curb FC head memorization
resnet.fc = nn.Sequential(
    nn.Dropout(0.6),
    nn.Linear(resnet.fc.in_features, 7)
)

model = resnet.to(device)

In [4]:
import numpy as np

# --- Mixup Helper Functions ---
def mixup_data(x, y, alpha=0.05):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- Setup Directory & Loss ---
os.makedirs("../../models", exist_ok=True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.08) # Slightly reduced label smoothing for sharper probabilities

# --- Balanced Optimizer Parameters ---
optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 1.2e-5, "weight_decay": 1e-3},
    {"params": resnet.layer4.parameters(), "lr": 1.2e-4, "weight_decay": 1e-3},
    {"params": resnet.fc.parameters(),     "lr": 3.5e-4, "weight_decay": 1e-2}
])

num_epochs = 40
scheduler  = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

patience         = 12
best_val_loss    = float("inf")
best_val_acc     = 0.0
patience_counter = 0

save_path = "../../models/best_rafdb_resnet18_enhanced.pth"

# --- Training Loop ---
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        inputs, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.05)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
        correct_a = (outputs.argmax(1) == targets_a).sum().item()
        correct_b = (outputs.argmax(1) == targets_b).sum().item()
        train_correct += (lam * correct_a + (1 - lam) * correct_b)

    scheduler.step()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    # --- Validation Loop with TTA (Test-Time Augmentation) ---
    model.eval()
    val_loss, val_correct = 0.0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            # 1. Forward pass on original images
            outputs_orig = model(images)
            loss = criterion(outputs_orig, labels)

            # 2. Forward pass on horizontally flipped images (TTA)
            images_flipped = torch.flip(images, dims=[3])
            outputs_flipped = model(images_flipped)

            # 3. Average prediction probabilities
            outputs_tta = (outputs_orig + outputs_flipped) / 2.0

            val_loss    += loss.item()
            val_correct += (outputs_tta.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    current_lr = scheduler.get_last_lr()[-1]
    print(f"Epoch [{epoch+1}/{num_epochs}] (LR: {current_lr:.6f})  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc (TTA): {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"  ✓ Best model saved to {save_path} (val loss: {val_loss:.4f}, val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(save_path))
print(f"Best enhanced ResNet-18 model loaded! Final Validation Accuracy: {best_val_acc:.2f}%")

Epoch [1/40] (LR: 0.000349)  Train Loss: 1.7188  Train Acc: 37.74%  |  Val Loss: 1.3989  Val Acc (TTA): 56.71%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.3989, val acc: 56.71%)
Epoch [2/40] (LR: 0.000348)  Train Loss: 1.4359  Train Acc: 51.33%  |  Val Loss: 1.3841  Val Acc (TTA): 54.43%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.3841, val acc: 54.43%)
Epoch [3/40] (LR: 0.000345)  Train Loss: 1.3127  Train Acc: 57.89%  |  Val Loss: 1.1592  Val Acc (TTA): 65.89%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.1592, val acc: 65.89%)
Epoch [4/40] (LR: 0.000341)  Train Loss: 1.2214  Train Acc: 62.28%  |  Val Loss: 1.1334  Val Acc (TTA): 67.52%
  ✓ Best model saved to ../../models/best_rafdb_resnet18_enhanced.pth (val loss: 1.1334, val acc: 67.52%)
Epoch [5/40] (LR: 0.000337)  Train Loss: 1.1649  Train Acc: 65.63%  |  Val Loss: 0.9907  Val Acc (TTA): 73.93%
  ✓ Best model saved 